# Experimentos Univariados v3 — Cloud (Vertex AI)

**Tesis MEC** — Grilla completa: 97 DGPs × T∈{25,50,100,200} × R=500  
**Horizonte por T:** T=25→H=6 (Corto) · T=50→H=18 (Corto+Medio) · T=100,200→H=24 (todos)  
**Métricas:** Bias, Varianza, RMSE, CRPS  
**Bloques:** Corto h=1–6 · Medio h=7–18 · Largo h=19–24  
**Logging:** dual stdout + `results/univariate_v3/run_YYYYMMDD_HHMMSS.log`  
**Resultados:** `results/univariate_v3/` — si existen se cargan sin re-simular

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import logging
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import torch
from IPython.display import display

from mectesis.dgp import (
    RandomWalk, SeasonalDGP,
    AR1ARCH, AR1GARCH,
    LocalLevelDGP, LocalTrendDGP, DampedTrendDGP, LocalLevelSeasonalDGP,
    ARpDGP, MAqDGP, ARMApqDGP, ARMApqWithTrendDGP,
    SETARDGp, LSTARDGp, ESTARDGp,
)
from mectesis.models import (
    ARIMAModel, ARIMAWithTrendModel, SARIMAModel,
    ARARCHModel, ARGARCHModel,
    ETSModel, ThetaModel, ChronosModel,
)
from mectesis.simulation import MonteCarloEngine

SEED    = 3649
H_BY_T  = {25: 6, 50: 18, 100: 24, 200: 24}  # horizonte por T total
H_MAX   = 24                                   # para definir los bloques
R_LIST  = [500]
T_LIST  = [25, 50, 100, 200]                   # longitud TOTAL de la serie
RESULTS = Path("results/univariate_v3")
RESULTS.mkdir(parents=True, exist_ok=True)

# ── Logging dual: celda de Jupyter + archivo .log ────────────────────────────
log_path = RESULTS / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger().info
log(f"Log en: {log_path}")

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)

device = "cuda" if torch.cuda.is_available() else "cpu"
log(f"Cargando Chronos-2 en {device} (puede tardar ~30 s la primera vez)...")
chronos = ChronosModel(device=device)
log("Chronos-2 listo.")

In [ ]:
# ─── Funciones auxiliares ────────────────────────────────────────────────────

def _cache_path(exp_id: str, T: int, R: int) -> Path:
    return RESULTS / f"exp_{exp_id.replace('.', '_')}_T{T}_R{R}.csv"


def _save_results(results: dict, path: Path):
    frames = []
    for mname, df in results.items():
        tmp = df.copy()
        tmp.insert(0, "model", mname)
        frames.append(tmp)
    pd.concat(frames, ignore_index=True).to_csv(path, index=False)


def _load_results(path: Path) -> dict:
    df = pd.read_csv(path)
    return {
        mname: grp.drop(columns="model").reset_index(drop=True)
        for mname, grp in df.groupby("model", sort=False)
    }


def run_exp(dgp, make_models_fn, dgp_params, exp_id,
            T_list=T_LIST, R_list=R_LIST, H_by_T=None, seed=SEED):
    """
    T_list: longitudes TOTALES de la serie.
    El engine hace y_train = y[:T-H] internamente.
    H_by_T: horizonte por T (por defecto H_BY_T global).
      T=25 → H=6  (Corto, T_train=19)
      T=50 → H=18 (Corto+Medio, T_train=32)
      T=100,200 → H=24 (todos los bloques)
    """
    if H_by_T is None:
        H_by_T = H_BY_T
    n_runs = len(T_list) * len(R_list)
    combos = ", ".join(
        f"(T={t}, H={H_by_T.get(t, H_MAX)}, R={r})"
        for t in T_list for r in R_list
    )
    log(f"Exp {exp_id}: {n_runs} ejecución(es) → {combos}")
    all_results = {}
    for T in T_list:
        h = H_by_T.get(T, H_MAX)
        for R in R_list:
            cache = _cache_path(exp_id, T, R)
            if cache.exists():
                log(f"  T={T} H={h}, R={R}: cargando {cache.name}")
                all_results[(T, R)] = _load_results(cache)
                continue
            log(f"  T={T} H={h}, R={R}: simulando...")
            dgp.rng = np.random.default_rng(seed)
            models = make_models_fn(T)
            engine = MonteCarloEngine(dgp, models, seed=seed)
            t0 = time.time()
            results = engine.run_monte_carlo(R, T, h, dgp_params, verbose=False)
            log(f"  T={T} H={h}, R={R}: OK ({time.time()-t0:.0f}s)")
            _save_results(results, cache)
            all_results[(T, R)] = results
    return all_results


# ─── Funciones v3 ────────────────────────────────────────────────────────────

BLOCK_DEFS = [("C", 1, 6), ("M", 7, 18), ("L", 19, 24)]
METRICS_V3  = ["bias", "variance", "rmse", "crps"]


def compute_blocks_v3(results_TR: dict) -> dict:
    out = {}
    for mname, df in results_TR.items():
        df_h = df[df["horizon"] != "avg_all"].copy()
        df_h["horizon"] = pd.to_numeric(df_h["horizon"], errors="coerce")
        blks = {}
        for blk, h1, h2 in BLOCK_DEFS:
            mask = (df_h["horizon"] >= h1) & (df_h["horizon"] <= h2)
            blks[blk] = df_h[mask].mean(numeric_only=True)
        out[mname] = blks
    return out


def build_grid_table(all_results: dict, classical_name: str,
                     chronos_name: str = "Chronos-2"):
    """
    Tabla ancha: una fila por (T, Modelo).
    best_rmse / best_crps: 'C' si clásico <= Chronos, 'T' si no.
    """
    rows = []
    for (T, R), res_TR in sorted(all_results.items()):
        blk_data = compute_blocks_v3(res_TR)
        cl_blks  = blk_data.get(classical_name, {})
        ch_blks  = blk_data.get(chronos_name, {})

        for mname, blks in blk_data.items():
            row = {"T": T, "Modelo": mname}
            for blk, h1, h2 in BLOCK_DEFS:
                s = blks.get(blk, pd.Series(dtype=float))
                for m in METRICS_V3:
                    row[f"{m}_{blk}"] = (
                        round(float(s[m]), 4)
                        if m in s.index and pd.notna(s[m]) else np.nan
                    )
                cl_s = cl_blks.get(blk, pd.Series(dtype=float))
                ch_s = ch_blks.get(blk, pd.Series(dtype=float))
                for m in ["rmse", "crps"]:
                    cv = float(cl_s[m]) if m in cl_s.index and pd.notna(cl_s[m]) else np.nan
                    hv = float(ch_s[m]) if m in ch_s.index and pd.notna(ch_s[m]) else np.nan
                    if np.isnan(cv) or np.isnan(hv):
                        row[f"best_{m}_{blk}"] = np.nan
                    else:
                        row[f"best_{m}_{blk}"] = "C" if cv <= hv else "T"
            rows.append(row)

    df_out = pd.DataFrame(rows).set_index(["T", "Modelo"])
    display(df_out.style.format(precision=4, na_rep="—"))


def plot_simulation_v3(dgp, models, dgp_params, title="", T_vis=100, seed=SEED):
    """1 realización del DGP: histórico + split + observado + forecasts superpuestos."""
    H_vis   = H_BY_T.get(T_vis, H_MAX)
    old_rng = dgp.rng
    dgp.rng = np.random.default_rng(seed + 99991)
    y       = dgp.simulate(T=T_vis, **dgp_params)
    dgp.rng = old_rng

    split   = T_vis - H_vis
    y_train = y[:split]
    y_test  = y[split:]
    t_train = np.arange(split)
    t_test  = np.arange(split, T_vis)

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(t_train, y_train, color="gray", lw=1.5, label="Histórico")
    ax.axvline(split - 0.5, color="black", ls="--", lw=1, alpha=0.6)
    ax.plot(t_test, y_test, color="black", lw=1.5, marker="o", ms=3, label="Observado")

    palette = ["steelblue", "darkorange", "seagreen", "purple"]
    for i, model in enumerate(models):
        try:
            model.fit(y_train)
            fcst = model.forecast(H_vis)
            c = palette[i % len(palette)]
            ax.plot(t_test, fcst, color=c, lw=1.5, ls="--", marker="s", ms=3, label=model.name)
            if getattr(model, "supports_intervals", False):
                lo, hi = model.forecast_intervals(H_vis, level=0.80)
                ax.fill_between(t_test, lo, hi, color=c, alpha=0.15)
        except Exception as e:
            log(f"  [plot_simulation_v3] {model.name} falló: {e}")

    ax.set(title=title, xlabel="t", ylabel="y")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

---
## Bloque A — ARMA sin tendencia (24 experimentos)

DGPs: AR(1–4) · MA(1–4) · ARMA(1,1) · ARMA(2,2) · ARMA(1,4) · ARMA(4,1)  
Modelo clásico: ARIMA(p,0,q) correctamente especificado vs Chronos-2

In [ ]:
# A.1 — AR(1) ρ=0.30
cl  = ARIMAModel((1, 0, 0))
dgp = ARpDGP(phis=[0.3], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.1")
log(f"\n============================================================\nA.1 — AR(1) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.1 — AR(1) ρ=0.30")

In [ ]:
# A.2 — AR(1) ρ=0.90
cl  = ARIMAModel((1, 0, 0))
dgp = ARpDGP(phis=[0.9], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.2")
log(f"\n============================================================\nA.2 — AR(1) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.2 — AR(1) ρ=0.90")

In [ ]:
# A.3 — AR(2) ρ=0.30
cl  = ARIMAModel((2, 0, 0))
dgp = ARpDGP(phis=[0.3, 0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.3")
log(f"\n============================================================\nA.3 — AR(2) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.3 — AR(2) ρ=0.30")

In [ ]:
# A.4 — AR(2) ρ=0.90
cl  = ARIMAModel((2, 0, 0))
dgp = ARpDGP(phis=[0.9, -0.2], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.4")
log(f"\n============================================================\nA.4 — AR(2) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.4 — AR(2) ρ=0.90")

In [ ]:
# A.5 — AR(3) ρ=0.30
cl  = ARIMAModel((3, 0, 0))
dgp = ARpDGP(phis=[0.3, 0.1, 0.05], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.5")
log(f"\n============================================================\nA.5 — AR(3) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(3, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.5 — AR(3) ρ=0.30")

In [ ]:
# A.6 — AR(3) ρ=0.90
cl  = ARIMAModel((3, 0, 0))
dgp = ARpDGP(phis=[0.9, -0.2, 0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.6")
log(f"\n============================================================\nA.6 — AR(3) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(3, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.6 — AR(3) ρ=0.90")

In [ ]:
# A.7 — AR(4) ρ=0.30
cl  = ARIMAModel((4, 0, 0))
dgp = ARpDGP(phis=[0.3, 0.1, 0.05, 0.02], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.7")
log(f"\n============================================================\nA.7 — AR(4) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.7 — AR(4) ρ=0.30")

In [ ]:
# A.8 — AR(4) ρ=0.90
cl  = ARIMAModel((4, 0, 0))
dgp = ARpDGP(phis=[0.9, -0.2, 0.1, -0.05], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.8")
log(f"\n============================================================\nA.8 — AR(4) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 0)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.8 — AR(4) ρ=0.90")

In [ ]:
# A.9 — MA(1) θ=0.30
cl  = ARIMAModel((0, 0, 1))
dgp = MAqDGP(thetas=[0.3], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.9")
log(f"\n============================================================\nA.9 — MA(1) θ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 1)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.9 — MA(1) θ=0.30")

In [ ]:
# A.10 — MA(1) θ=0.90
cl  = ARIMAModel((0, 0, 1))
dgp = MAqDGP(thetas=[0.9], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.10")
log(f"\n============================================================\nA.10 — MA(1) θ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 1)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.10 — MA(1) θ=0.90")

In [ ]:
# A.11 — MA(2) θ=0.30
cl  = ARIMAModel((0, 0, 2))
dgp = MAqDGP(thetas=[0.3, 0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.11")
log(f"\n============================================================\nA.11 — MA(2) θ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 2)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.11 — MA(2) θ=0.30")

In [ ]:
# A.12 — MA(2) θ=0.90
cl  = ARIMAModel((0, 0, 2))
dgp = MAqDGP(thetas=[0.9, 0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.12")
log(f"\n============================================================\nA.12 — MA(2) θ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 2)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.12 — MA(2) θ=0.90")

In [ ]:
# A.13 — MA(3) θ=0.30
cl  = ARIMAModel((0, 0, 3))
dgp = MAqDGP(thetas=[0.3, 0.1, -0.05], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.13")
log(f"\n============================================================\nA.13 — MA(3) θ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 3)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.13 — MA(3) θ=0.30")

In [ ]:
# A.14 — MA(3) θ=0.90
cl  = ARIMAModel((0, 0, 3))
dgp = MAqDGP(thetas=[0.9, 0.1, -0.05], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.14")
log(f"\n============================================================\nA.14 — MA(3) θ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 3)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.14 — MA(3) θ=0.90")

In [ ]:
# A.15 — MA(4) θ=0.30
cl  = ARIMAModel((0, 0, 4))
dgp = MAqDGP(thetas=[0.3, 0.1, -0.05, 0.02], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.15")
log(f"\n============================================================\nA.15 — MA(4) θ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 4)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.15 — MA(4) θ=0.30")

In [ ]:
# A.16 — MA(4) θ=0.90
cl  = ARIMAModel((0, 0, 4))
dgp = MAqDGP(thetas=[0.9, 0.1, -0.05, 0.02], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.16")
log(f"\n============================================================\nA.16 — MA(4) θ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 4)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.16 — MA(4) θ=0.90")

In [ ]:
# A.17 — ARMA(1,1) ρ=0.30
cl  = ARIMAModel((1, 0, 1))
dgp = ARMApqDGP(phis=[0.3], thetas=[0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.17")
log(f"\n============================================================\nA.17 — ARMA(1,1) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 1)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.17 — ARMA(1,1) ρ=0.30")

In [ ]:
# A.18 — ARMA(1,1) ρ=0.90
cl  = ARIMAModel((1, 0, 1))
dgp = ARMApqDGP(phis=[0.9], thetas=[0.3], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.18")
log(f"\n============================================================\nA.18 — ARMA(1,1) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 1)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.18 — ARMA(1,1) ρ=0.90")

In [ ]:
# A.19 — ARMA(2,2) ρ=0.30
cl  = ARIMAModel((2, 0, 2))
dgp = ARMApqDGP(phis=[0.3, 0.1], thetas=[0.1, 0.05], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.19")
log(f"\n============================================================\nA.19 — ARMA(2,2) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 2)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.19 — ARMA(2,2) ρ=0.30")

In [ ]:
# A.20 — ARMA(2,2) ρ=0.90
cl  = ARIMAModel((2, 0, 2))
dgp = ARMApqDGP(phis=[0.9, -0.2], thetas=[0.3, -0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.20")
log(f"\n============================================================\nA.20 — ARMA(2,2) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 2)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.20 — ARMA(2,2) ρ=0.90")

In [ ]:
# A.21 — ARMA(1,4) ρ=0.30
cl  = ARIMAModel((1, 0, 4))
dgp = ARMApqDGP(phis=[0.3], thetas=[0.1, 0.05, -0.03, 0.01], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.21")
log(f"\n============================================================\nA.21 — ARMA(1,4) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 4)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.21 — ARMA(1,4) ρ=0.30")

In [ ]:
# A.22 — ARMA(1,4) ρ=0.90
cl  = ARIMAModel((1, 0, 4))
dgp = ARMApqDGP(phis=[0.9], thetas=[0.3, -0.1, 0.05, -0.02], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.22")
log(f"\n============================================================\nA.22 — ARMA(1,4) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 4)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.22 — ARMA(1,4) ρ=0.90")

In [ ]:
# A.23 — ARMA(4,1) ρ=0.30
cl  = ARIMAModel((4, 0, 1))
dgp = ARMApqDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[0.1], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.23")
log(f"\n============================================================\nA.23 — ARMA(4,1) ρ=0.30\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 1)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.23 — ARMA(4,1) ρ=0.30")

In [ ]:
# A.24 — ARMA(4,1) ρ=0.90
cl  = ARIMAModel((4, 0, 1))
dgp = ARMApqDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[0.3], sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.24")
log(f"\n============================================================\nA.24 — ARMA(4,1) ρ=0.90\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 1)")
plot_simulation_v3(dgp, [cl, chronos], {}, title="A.24 — ARMA(4,1) ρ=0.90")

---
## Bloque B — ARMA con tendencia determinística (48 experimentos)

DGP: `Y_t = α + δ·t + ARMA_t`  
Tendencia leve: δ=0.02 (B.1–B.24) | Tendencia fuerte: δ=0.10 (B.25–B.48)  
Modelo clásico: ARIMA(p,0,q)+trend (trend='ct')

In [ ]:
# B.1 — AR(1) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.1")
log(f"\n============================================================\nB.1 — AR(1) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.1 — AR(1) ρ=0.30 (δ=0.02)")

In [ ]:
# B.2 — AR(1) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.2")
log(f"\n============================================================\nB.2 — AR(1) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.2 — AR(1) ρ=0.90 (δ=0.02)")

In [ ]:
# B.3 — AR(2) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.3")
log(f"\n============================================================\nB.3 — AR(2) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.3 — AR(2) ρ=0.30 (δ=0.02)")

In [ ]:
# B.4 — AR(2) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.4")
log(f"\n============================================================\nB.4 — AR(2) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.4 — AR(2) ρ=0.90 (δ=0.02)")

In [ ]:
# B.5 — AR(3) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.5")
log(f"\n============================================================\nB.5 — AR(3) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.5 — AR(3) ρ=0.30 (δ=0.02)")

In [ ]:
# B.6 — AR(3) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.6")
log(f"\n============================================================\nB.6 — AR(3) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.6 — AR(3) ρ=0.90 (δ=0.02)")

In [ ]:
# B.7 — AR(4) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.7")
log(f"\n============================================================\nB.7 — AR(4) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.7 — AR(4) ρ=0.30 (δ=0.02)")

In [ ]:
# B.8 — AR(4) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.8")
log(f"\n============================================================\nB.8 — AR(4) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.8 — AR(4) ρ=0.90 (δ=0.02)")

In [ ]:
# B.9 — MA(1) θ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.9")
log(f"\n============================================================\nB.9 — MA(1) θ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.9 — MA(1) θ=0.30 (δ=0.02)")

In [ ]:
# B.10 — MA(1) θ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.10")
log(f"\n============================================================\nB.10 — MA(1) θ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.10 — MA(1) θ=0.90 (δ=0.02)")

In [ ]:
# B.11 — MA(2) θ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.11")
log(f"\n============================================================\nB.11 — MA(2) θ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.11 — MA(2) θ=0.30 (δ=0.02)")

In [ ]:
# B.12 — MA(2) θ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.12")
log(f"\n============================================================\nB.12 — MA(2) θ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.12 — MA(2) θ=0.90 (δ=0.02)")

In [ ]:
# B.13 — MA(3) θ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.13")
log(f"\n============================================================\nB.13 — MA(3) θ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.13 — MA(3) θ=0.30 (δ=0.02)")

In [ ]:
# B.14 — MA(3) θ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.14")
log(f"\n============================================================\nB.14 — MA(3) θ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.14 — MA(3) θ=0.90 (δ=0.02)")

In [ ]:
# B.15 — MA(4) θ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05, 0.02], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.15")
log(f"\n============================================================\nB.15 — MA(4) θ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.15 — MA(4) θ=0.30 (δ=0.02)")

In [ ]:
# B.16 — MA(4) θ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05, 0.02], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.16")
log(f"\n============================================================\nB.16 — MA(4) θ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.16 — MA(4) θ=0.90 (δ=0.02)")

In [ ]:
# B.17 — ARMA(1,1) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.17")
log(f"\n============================================================\nB.17 — ARMA(1,1) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.17 — ARMA(1,1) ρ=0.30 (δ=0.02)")

In [ ]:
# B.18 — ARMA(1,1) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.18")
log(f"\n============================================================\nB.18 — ARMA(1,1) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.18 — ARMA(1,1) ρ=0.90 (δ=0.02)")

In [ ]:
# B.19 — ARMA(2,2) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[0.1, 0.05], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.19")
log(f"\n============================================================\nB.19 — ARMA(2,2) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.19 — ARMA(2,2) ρ=0.30 (δ=0.02)")

In [ ]:
# B.20 — ARMA(2,2) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[0.3, -0.1], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.20")
log(f"\n============================================================\nB.20 — ARMA(2,2) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.20 — ARMA(2,2) ρ=0.90 (δ=0.02)")

In [ ]:
# B.21 — ARMA(1,4) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1, 0.05, -0.03, 0.01], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.21")
log(f"\n============================================================\nB.21 — ARMA(1,4) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.21 — ARMA(1,4) ρ=0.30 (δ=0.02)")

In [ ]:
# B.22 — ARMA(1,4) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3, -0.1, 0.05, -0.02], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.22")
log(f"\n============================================================\nB.22 — ARMA(1,4) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.22 — ARMA(1,4) ρ=0.90 (δ=0.02)")

In [ ]:
# B.23 — ARMA(4,1) ρ=0.30  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[0.1], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.23")
log(f"\n============================================================\nB.23 — ARMA(4,1) ρ=0.30 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.23 — ARMA(4,1) ρ=0.30 (δ=0.02)")

In [ ]:
# B.24 — ARMA(4,1) ρ=0.90  [tendencia leve δ=0.02]
cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[0.3], delta=0.02, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.24")
log(f"\n============================================================\nB.24 — ARMA(4,1) ρ=0.90 (δ=0.02)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.24 — ARMA(4,1) ρ=0.90 (δ=0.02)")

In [ ]:
# B.25 — AR(1) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.25")
log(f"\n============================================================\nB.25 — AR(1) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.25 — AR(1) ρ=0.30 (δ=0.1)")

In [ ]:
# B.26 — AR(1) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.26")
log(f"\n============================================================\nB.26 — AR(1) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.26 — AR(1) ρ=0.90 (δ=0.1)")

In [ ]:
# B.27 — AR(2) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.27")
log(f"\n============================================================\nB.27 — AR(2) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.27 — AR(2) ρ=0.30 (δ=0.1)")

In [ ]:
# B.28 — AR(2) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.28")
log(f"\n============================================================\nB.28 — AR(2) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.28 — AR(2) ρ=0.90 (δ=0.1)")

In [ ]:
# B.29 — AR(3) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.29")
log(f"\n============================================================\nB.29 — AR(3) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.29 — AR(3) ρ=0.30 (δ=0.1)")

In [ ]:
# B.30 — AR(3) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.30")
log(f"\n============================================================\nB.30 — AR(3) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.30 — AR(3) ρ=0.90 (δ=0.1)")

In [ ]:
# B.31 — AR(4) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.31")
log(f"\n============================================================\nB.31 — AR(4) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.31 — AR(4) ρ=0.30 (δ=0.1)")

In [ ]:
# B.32 — AR(4) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.32")
log(f"\n============================================================\nB.32 — AR(4) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.32 — AR(4) ρ=0.90 (δ=0.1)")

In [ ]:
# B.33 — MA(1) θ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.33")
log(f"\n============================================================\nB.33 — MA(1) θ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.33 — MA(1) θ=0.30 (δ=0.1)")

In [ ]:
# B.34 — MA(1) θ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.34")
log(f"\n============================================================\nB.34 — MA(1) θ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.34 — MA(1) θ=0.90 (δ=0.1)")

In [ ]:
# B.35 — MA(2) θ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.35")
log(f"\n============================================================\nB.35 — MA(2) θ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.35 — MA(2) θ=0.30 (δ=0.1)")

In [ ]:
# B.36 — MA(2) θ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.36")
log(f"\n============================================================\nB.36 — MA(2) θ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.36 — MA(2) θ=0.90 (δ=0.1)")

In [ ]:
# B.37 — MA(3) θ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.37")
log(f"\n============================================================\nB.37 — MA(3) θ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.37 — MA(3) θ=0.30 (δ=0.1)")

In [ ]:
# B.38 — MA(3) θ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.38")
log(f"\n============================================================\nB.38 — MA(3) θ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.38 — MA(3) θ=0.90 (δ=0.1)")

In [ ]:
# B.39 — MA(4) θ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05, 0.02], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.39")
log(f"\n============================================================\nB.39 — MA(4) θ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.39 — MA(4) θ=0.30 (δ=0.1)")

In [ ]:
# B.40 — MA(4) θ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05, 0.02], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.40")
log(f"\n============================================================\nB.40 — MA(4) θ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.40 — MA(4) θ=0.90 (δ=0.1)")

In [ ]:
# B.41 — ARMA(1,1) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.41")
log(f"\n============================================================\nB.41 — ARMA(1,1) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.41 — ARMA(1,1) ρ=0.30 (δ=0.1)")

In [ ]:
# B.42 — ARMA(1,1) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.42")
log(f"\n============================================================\nB.42 — ARMA(1,1) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.42 — ARMA(1,1) ρ=0.90 (δ=0.1)")

In [ ]:
# B.43 — ARMA(2,2) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[0.1, 0.05], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.43")
log(f"\n============================================================\nB.43 — ARMA(2,2) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.43 — ARMA(2,2) ρ=0.30 (δ=0.1)")

In [ ]:
# B.44 — ARMA(2,2) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[0.3, -0.1], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.44")
log(f"\n============================================================\nB.44 — ARMA(2,2) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.44 — ARMA(2,2) ρ=0.90 (δ=0.1)")

In [ ]:
# B.45 — ARMA(1,4) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1, 0.05, -0.03, 0.01], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.45")
log(f"\n============================================================\nB.45 — ARMA(1,4) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.45 — ARMA(1,4) ρ=0.30 (δ=0.1)")

In [ ]:
# B.46 — ARMA(1,4) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3, -0.1, 0.05, -0.02], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.46")
log(f"\n============================================================\nB.46 — ARMA(1,4) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.46 — ARMA(1,4) ρ=0.90 (δ=0.1)")

In [ ]:
# B.47 — ARMA(4,1) ρ=0.30  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[0.1], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.47")
log(f"\n============================================================\nB.47 — ARMA(4,1) ρ=0.30 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.47 — ARMA(4,1) ρ=0.30 (δ=0.1)")

In [ ]:
# B.48 — ARMA(4,1) ρ=0.90  [tendencia fuerte δ=0.1]
cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[0.3], delta=0.1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.48")
log(f"\n============================================================\nB.48 — ARMA(4,1) ρ=0.90 (δ=0.1)\n============================================================")
build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
plot_simulation_v3(dgp, [cl, chronos], {}, title="B.48 — ARMA(4,1) ρ=0.90 (δ=0.1)")

---
## Bloque C — Random Walk (3 experimentos)

DGP: `Y_t = drift + Y_{t-1} + ε_t`  
Modelo clásico: ARIMA(0,1,0)

In [ ]:
# C.1 — RW sin drift
cl  = ARIMAModel((0, 1, 0))
dgp = RandomWalk(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {"drift": 0.0, "sigma": 1.0}, exp_id="C.1")
log("\n" + "="*60 + "\nC.1 — RW sin drift\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"drift": 0.0, "sigma": 1.0}, title="C.1 — Random Walk sin drift")

In [ ]:
# C.2 — RW drift leve (δ=0.05)
cl  = ARIMAModel((0, 1, 0))
dgp = RandomWalk(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {"drift": 0.05, "sigma": 1.0}, exp_id="C.2")
log("\n" + "="*60 + "\nC.2 — RW drift leve (δ=0.05)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"drift": 0.05, "sigma": 1.0}, title="C.2 — Random Walk drift leve (δ=0.05)")

In [ ]:
# C.3 — RW drift fuerte (δ=0.20)
cl  = ARIMAModel((0, 1, 0))
dgp = RandomWalk(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {"drift": 0.20, "sigma": 1.0}, exp_id="C.3")
log("\n" + "="*60 + "\nC.3 — RW drift fuerte (δ=0.20)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"drift": 0.20, "sigma": 1.0}, title="C.3 — Random Walk drift fuerte (δ=0.20)")

---
## Bloque D — Volatilidad condicional: ARCH/GARCH (4 experimentos)

DGP: AR(1) en la media + proceso de varianza condicional  
Modelo clásico: AR+ARCH o AR+GARCH correctamente especificado

In [ ]:
# D.1 — AR(1)-ARCH(1) leve  (α=0.10)
cl  = ARARCHModel(ar_lags=1, p=1)
dgp = AR1ARCH(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.5, "omega": 0.5, "alpha": 0.10}, exp_id="D.1")
log("\n" + "="*60 + "\nD.1 — AR(1)-ARCH(1) leve\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.5, "alpha": 0.10}, title="D.1 — AR(1)-ARCH(1) leve (α=0.10)")

In [ ]:
# D.2 — AR(1)-ARCH(1) fuerte  (α=0.50)
cl  = ARARCHModel(ar_lags=1, p=1)
dgp = AR1ARCH(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.5, "omega": 0.5, "alpha": 0.50}, exp_id="D.2")
log("\n" + "="*60 + "\nD.2 — AR(1)-ARCH(1) fuerte\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.5, "alpha": 0.50}, title="D.2 — AR(1)-ARCH(1) fuerte (α=0.50)")

In [ ]:
# D.3 — AR(1)-GARCH(1,1) baja persistencia  (α+β=0.50)
cl  = ARGARCHModel(ar_lags=1, p=1, q=1)
dgp = AR1GARCH(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.5, "omega": 0.5, "alpha": 0.10, "beta": 0.40}, exp_id="D.3")
log("\n" + "="*60 + "\nD.3 — AR(1)-GARCH(1,1) baja persistencia\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.5, "alpha": 0.10, "beta": 0.40}, title="D.3 — AR(1)-GARCH(1,1) baja persistencia (α+β=0.50)")

In [ ]:
# D.4 — AR(1)-GARCH(1,1) alta persistencia  (α+β=0.95)
cl  = ARGARCHModel(ar_lags=1, p=1, q=1)
dgp = AR1GARCH(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.5, "omega": 0.1, "alpha": 0.10, "beta": 0.85}, exp_id="D.4")
log("\n" + "="*60 + "\nD.4 — AR(1)-GARCH(1,1) alta persistencia\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.1, "alpha": 0.10, "beta": 0.85}, title="D.4 — AR(1)-GARCH(1,1) alta persistencia (α+β=0.95)")

---
## Bloque E — ETS y Theta (8 experimentos)

DGPs de espacio de estados: nivel local, tendencia, estacionalidad  
Modelos clásicos: ETS(A,T,S) y Theta

In [ ]:
# E.1 — Local Level  ETS(A,N,N)
cl  = ETSModel()
dgp = LocalLevelDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"sigma_eps": 1.0, "sigma_eta": 0.10}, exp_id="E.1")
log("\n" + "="*60 + "\nE.1 — Local Level ETS(A,N,N)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.10}, title="E.1 — Local Level ETS(A,N,N)")

In [ ]:
# E.2 — Local Linear Trend leve  ETS(A,A,N)  σ_ζ=0.05
cl  = ETSModel(trend="add")
dgp = LocalTrendDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.05, "b0": 0.1}, exp_id="E.2")
log("\n" + "="*60 + "\nE.2 — LLT leve ETS(A,A,N)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.05, "b0": 0.1}, title="E.2 — Local Linear Trend leve (σ_ζ=0.05)")

In [ ]:
# E.3 — Local Linear Trend fuerte  ETS(A,A,N)  σ_ζ=0.20
cl  = ETSModel(trend="add")
dgp = LocalTrendDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.20, "b0": 0.5}, exp_id="E.3")
log("\n" + "="*60 + "\nE.3 — LLT fuerte ETS(A,A,N)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.20, "b0": 0.5}, title="E.3 — Local Linear Trend fuerte (σ_ζ=0.20)")

In [ ]:
# E.4 — Damped Trend  ETS(A,Ad,N)  φ_d=0.90
cl  = ETSModel(trend="add", damped_trend=True)
dgp = DampedTrendDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.9, "sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.1}, exp_id="E.4")
log("\n" + "="*60 + "\nE.4 — Damped Trend ETS(A,Ad,N)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.9, "sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.1}, title="E.4 — Damped Trend ETS(A,Ad,N)")

In [ ]:
# E.5 — Seasonal Aditiva s=12  ETS(A,N,A)  (sin tendencia: b0=0, sigma_zeta=0)
cl  = ETSModel(seasonal="add", seasonal_periods=12)
dgp = LocalLevelSeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1,
               "sigma_zeta": 0.0, "sigma_omega": 0.05, "b0": 0.0},
              exp_id="E.5", T_list=[50, 100, 200])  # T=25 muy corto para s=12
log("\n" + "="*60 + "\nE.5 — Seasonal Aditiva s=12 ETS(A,N,A)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1, "sigma_zeta": 0.0, "sigma_omega": 0.05, "b0": 0.0}, title="E.5 — Seasonal Aditiva s=12 ETS(A,N,A)")

In [ ]:
# E.6 — Trend + Seasonal s=12  ETS(A,A,A)
cl  = ETSModel(trend="add", seasonal="add", seasonal_periods=12)
dgp = LocalLevelSeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1,
               "sigma_zeta": 0.05, "sigma_omega": 0.05, "b0": 0.1},
              exp_id="E.6", T_list=[50, 100, 200])
log("\n" + "="*60 + "\nE.6 — Trend+Seasonal s=12 ETS(A,A,A)\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1, "sigma_zeta": 0.05, "sigma_omega": 0.05, "b0": 0.1}, title="E.6 — Trend+Seasonal s=12 ETS(A,A,A)")

In [ ]:
# E.7 — Theta leve  (tendencia suave b0=0.10, σ_ζ=0.01)
cl  = ThetaModel()
dgp = LocalTrendDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.01, "b0": 0.10}, exp_id="E.7")
log("\n" + "="*60 + "\nE.7 — Theta leve\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.01, "b0": 0.10}, title="E.7 — Theta leve (b0=0.10, σ_ζ=0.01)")

In [ ]:
# E.8 — Theta fuerte  (tendencia marcada b0=0.50, σ_ζ=0.10)
cl  = ThetaModel()
dgp = LocalTrendDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.10, "b0": 0.50}, exp_id="E.8")
log("\n" + "="*60 + "\nE.8 — Theta fuerte\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.10, "b0": 0.50}, title="E.8 — Theta fuerte (b0=0.50, σ_ζ=0.10)")

---
## Bloque F — SARIMA (6 experimentos)

DGP: SAR(1)(1)_s estacionario · (1-L)(1-L^s) integrado  
Períodos estacionales: s=4 (trimestral) · s=12 (mensual)

In [ ]:
# F.1 — SAR(1)(1)_4 baja persist.
cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 4))
dgp = SeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.3, "Phi": 0.3, "s": 4, "sigma": 1.0, "integrated": False},
              exp_id="F.1", T_list=T_LIST)
log("\n" + "="*60 + "\nF.1 — SAR(1)(1)_4 baja persist.\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.3, "Phi": 0.3, "s": 4, "sigma": 1.0, "integrated": False}, title="F.1 — SAR(1)(1)_4 baja persist.")

In [ ]:
# F.2 — SAR(1)(1)_4 alta persist.
cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 4))
dgp = SeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.9, "Phi": 0.6, "s": 4, "sigma": 1.0, "integrated": False},
              exp_id="F.2", T_list=T_LIST)
log("\n" + "="*60 + "\nF.2 — SAR(1)(1)_4 alta persist.\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.9, "Phi": 0.6, "s": 4, "sigma": 1.0, "integrated": False}, title="F.2 — SAR(1)(1)_4 alta persist.")

In [ ]:
# F.3 — SAR(1)(1)_12 baja persist.
cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 12))
dgp = SeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.3, "Phi": 0.3, "s": 12, "sigma": 1.0, "integrated": False},
              exp_id="F.3", T_list=[50, 100, 200])
log("\n" + "="*60 + "\nF.3 — SAR(1)(1)_12 baja persist.\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.3, "Phi": 0.3, "s": 12, "sigma": 1.0, "integrated": False}, title="F.3 — SAR(1)(1)_12 baja persist.")

In [ ]:
# F.4 — SAR(1)(1)_12 alta persist.
cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 12))
dgp = SeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"phi": 0.9, "Phi": 0.6, "s": 12, "sigma": 1.0, "integrated": False},
              exp_id="F.4", T_list=[50, 100, 200])
log("\n" + "="*60 + "\nF.4 — SAR(1)(1)_12 alta persist.\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.9, "Phi": 0.6, "s": 12, "sigma": 1.0, "integrated": False}, title="F.4 — SAR(1)(1)_12 alta persist.")

In [ ]:
# F.5 — (1-L)(1-L^4) integrado
cl  = SARIMAModel(order=(0, 1, 0), seasonal_order=(0, 1, 0, 4))
dgp = SeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"s": 4, "sigma": 1.0, "integrated": True},
              exp_id="F.5", T_list=T_LIST)
log("\n" + "="*60 + "\nF.5 — (1-L)(1-L^4) integrado\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"s": 4, "sigma": 1.0, "integrated": True}, title="F.5 — (1-L)(1-L^4) integrado")

In [ ]:
# F.6 — (1-L)(1-L^12) integrado
cl  = SARIMAModel(order=(0, 1, 0), seasonal_order=(0, 1, 0, 12))
dgp = SeasonalDGP(seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
              {"s": 12, "sigma": 1.0, "integrated": True},
              exp_id="F.6", T_list=[50, 100, 200])
log("\n" + "="*60 + "\nF.6 — (1-L)(1-L^12) integrado\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {"s": 12, "sigma": 1.0, "integrated": True}, title="F.6 — (1-L)(1-L^12) integrado")

---
## Bloque G — No linealidad de umbral: SETAR / LSTAR / ESTAR (4 experimentos)

Benchmark clásico: ARIMA(1,0,0) — **lineal misspecificado**  
Pregunta: ¿detecta Chronos-2 la no-linealidad que ARIMA no puede capturar?  
- **SETAR**: umbral determinístico y observable (vs MS-AR latente ya implementado)  
- **LSTAR**: transición suave asimétrica (ciclos de negocios)  
- **ESTAR**: transición suave simétrica (tipos de cambio con bandas — canónico para Argentina)

In [ ]:
# G.1 — SETAR(2;1) baja persistencia  φ₁=0.30, φ₂=-0.30, k=0
cl  = ARIMAModel((1, 0, 0))
dgp = SETARDGp(phi1=0.30, phi2=-0.30, threshold=0.0, delay=1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.1")
log("\n" + "="*60 + "\nG.1 — SETAR(2;1) baja persistencia\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {}, title="G.1 — SETAR(2;1) φ₁=0.30 / φ₂=-0.30")

In [ ]:
# G.2 — SETAR(2;1) alta persistencia  φ₁=0.90, φ₂=-0.50, k=0
cl  = ARIMAModel((1, 0, 0))
dgp = SETARDGp(phi1=0.90, phi2=-0.50, threshold=0.0, delay=1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.2")
log("\n" + "="*60 + "\nG.2 — SETAR(2;1) alta persistencia\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {}, title="G.2 — SETAR(2;1) φ₁=0.90 / φ₂=-0.50")

In [ ]:
# G.3 — LSTAR(1) asimétrico  φ₁=0.30, φ₂=0.90, γ=2, c=0
#   G≈0 (régimen bajo): persistencia leve | G≈1 (régimen alto): persistencia alta
cl  = ARIMAModel((1, 0, 0))
dgp = LSTARDGp(phi1=0.30, phi2=0.90, gamma=2.0, c=0.0, delay=1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.3")
log("\n" + "="*60 + "\nG.3 — LSTAR(1) asimétrico\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {}, title="G.3 — LSTAR(1) φ₁=0.30 / φ₂=0.90 (γ=2)")

In [ ]:
# G.4 — ESTAR(1) simétrico  φ₁=0.90, φ₂=0.10, γ=1, c=0
#   G≈0 (cerca del equilibrio): alta persistencia | G≈1 (lejos): rápida reversión
#   Modelo canónico para tipos de cambio con bandas
cl  = ARIMAModel((1, 0, 0))
dgp = ESTARDGp(phi1=0.90, phi2=0.10, gamma=1.0, c=0.0, delay=1, sigma=1.0, seed=SEED)
res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.4")
log("\n" + "="*60 + "\nG.4 — ESTAR(1) simétrico\n" + "="*60)
build_grid_table(res, classical_name=cl.name)
plot_simulation_v3(dgp, [cl, chronos], {}, title="G.4 — ESTAR(1) φ₁=0.90 / φ₂=0.10 (γ=1)")

---
## Resumen

| Bloque | DGPs | T | R | Total series |
|--------|------|---|---|-------------|
| A — ARMA sin tendencia | 24 | 4 | 500 | 48,000 |
| B — ARMA con tendencia | 48 | 4 | 500 | 96,000 |
| C — Random Walk | 3 | 4 | 500 | 6,000 |
| D — ARCH/GARCH | 4 | 4 | 500 | 8,000 |
| E — ETS/Theta | 6* | 4 | 500 | ~12,000 |
| F — SARIMA | 6* | ≤4 | 500 | ~10,000 |
| G — SETAR/LSTAR/ESTAR | 4 | 4 | 500 | 8,000 |
| **Total** | **95** | — | — | **~188,000** |

*Algunos experimentos usan T_list reducido (s=12 requiere T≥50)